In [1]:
import pandas as pd 
import numpy as np
import gensim
from sklearn.feature_extraction.text import CountVectorizer, TfidfVectorizer
import nltk
from nltk.stem import PorterStemmer, WordNetLemmatizer
from nltk.corpus import stopwords
import re
from gensim.models import Word2Vec, KeyedVectors
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split
from tqdm import tqdm

In [2]:
nltk.download('wordnet')
!unzip /usr/share/nltk_data/corpora/wordnet.zip -d /usr/share/nltk_data/corpora/

[nltk_data] Downloading package wordnet to /usr/share/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
Archive:  /usr/share/nltk_data/corpora/wordnet.zip
   creating: /usr/share/nltk_data/corpora/wordnet/
  inflating: /usr/share/nltk_data/corpora/wordnet/lexnames  
  inflating: /usr/share/nltk_data/corpora/wordnet/data.verb  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.adv  
  inflating: /usr/share/nltk_data/corpora/wordnet/adv.exc  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.verb  
  inflating: /usr/share/nltk_data/corpora/wordnet/cntlist.rev  
  inflating: /usr/share/nltk_data/corpora/wordnet/data.adj  
  inflating: /usr/share/nltk_data/corpora/wordnet/index.adj  
  inflating: /usr/share/nltk_data/corpora/wordnet/LICENSE  
  inflating: /usr/share/nltk_data/corpora/wordnet/citation.bib  
  inflating: /usr/share/nltk_data/corpora/wordnet/noun.exc  
  inflating: /usr/share/nltk_data/corpora/wordnet/verb.exc  
  inflating: /usr/share/nltk_data/co

In [3]:
spam_dataset = pd.read_csv("/kaggle/input/spam-email-classification/email.csv")

In [4]:
spam_dataset.describe()

,Category,Message
count,5573,5573
unique,3,5158
top,ham,"Sorry, I'll call later"
freq,4825,30


In [5]:
spam_dataset.head()

,Category,Message
0,ham,"Go until jurong point, crazy.. Available only ..."
1,ham,Ok lar... Joking wif u oni...
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...
3,ham,U dun say so early hor... U c already then say...
4,ham,"Nah I don't think he goes to usf, he lives aro..."


In [6]:
spam_dataset["Category"].unique()

array(['ham', 'spam', '{"mode":"full"'], dtype=object)

In [7]:
spam_dataset['is_spam'] = spam_dataset['Category'].map(lambda x : 1 if x == "spam" else 0)

In [8]:
spam_dataset.head()

,Category,Message,is_spam
0,ham,"Go until jurong point, crazy.. Available only ...",0
1,ham,Ok lar... Joking wif u oni...,0
2,spam,Free entry in 2 a wkly comp to win FA Cup fina...,1
3,ham,U dun say so early hor... U c already then say...,0
4,ham,"Nah I don't think he goes to usf, he lives aro...",0


In [9]:
spam_dataset["is_spam"].sum()/ spam_dataset["is_spam"].shape[0]

0.13403911717207967

In [10]:
nltk.sent_tokenize(spam_dataset['Message'].loc[0])

['Go until jurong point, crazy..',
 'Available only in bugis n great world la e buffet... Cine there got amore wat...']

In [11]:
wnl = WordNetLemmatizer()
def preprocess_msg_lemmatize(sentence):
    sentence = re.sub(r"[^a-zA-Z0-9]", " ", sentence)
    words = nltk.word_tokenize(sentence)
    sentence = " ".join([wnl.lemmatize(word).lower() for word in words if word not in set(stopwords.words('english'))])
    return sentence
stemmer =PorterStemmer()
def preprocess_msg_stemmer(sentence):
    sentence = re.sub(r"[^a-zA-Z0-9]", " ", sentence)
    words = nltk.word_tokenize(sentence)
    sentence = " ".join([stemmer.stem(word).lower() for word in words if word not in set(stopwords.words('english'))])
    return sentence
    

In [12]:
spam_dataset['lemmatized_msg'] = spam_dataset['Message'].apply(preprocess_msg_lemmatize)
spam_dataset['stemmed_msg'] = spam_dataset['Message'].apply(preprocess_msg_stemmer)

In [13]:
#bag of words
#tf-idf
#word2vec (cbow, skip_gram)
l_bow = CountVectorizer(max_features=2000)
s_bow = CountVectorizer(max_features=2000)
tf_idf_bow = TfidfVectorizer(max_features=2000, ngram_range=(1,2))
l_X = l_bow.fit_transform(spam_dataset['lemmatized_msg'])
s_X = s_bow.fit_transform(spam_dataset['stemmed_msg'])
t_X = tf_idf_bow.fit_transform(spam_dataset['lemmatized_msg'])

In [14]:
l_X, s_X, t_X

(<5573x2000 sparse matrix of type '<class 'numpy.int64'>'
 	with 42550 stored elements in Compressed Sparse Row format>,
 <5573x2000 sparse matrix of type '<class 'numpy.int64'>'
 	with 44059 stored elements in Compressed Sparse Row format>,
 <5573x2000 sparse matrix of type '<class 'numpy.float64'>'
 	with 45477 stored elements in Compressed Sparse Row format>)

In [15]:
Y = spam_dataset['is_spam']
Y.shape

(5573,)

In [16]:
x_train, x_test, y_train, y_test = train_test_split(l_X,Y, test_size=0.2, random_state=42)
classifier = RandomForestClassifier()

classifier.fit(x_train, y_train)
classifier.score(x_test, y_test)

0.9811659192825112

In [17]:
x_train, x_test, y_train, y_test = train_test_split(s_X,Y, test_size=0.2, random_state=42)
classifier = RandomForestClassifier()

classifier.fit(x_train, y_train)
classifier.score(x_test, y_test)

0.9802690582959641

In [18]:
x_train, x_test, y_train, y_test = train_test_split(t_X, Y, test_size=0.2, random_state=42)
classifier = RandomForestClassifier()

classifier.fit(x_train, y_train)
classifier.score(x_test, y_test)

0.9766816143497757

In [19]:
print(t_X[0])

  (0, 1866)	0.2436582621942033
  (0, 741)	0.20215950426192347
  (0, 370)	0.3684027085445512
  (0, 942)	0.3684027085445512
  (0, 1949)	0.292336946168411
  (0, 745)	0.24089423163690768
  (0, 281)	0.3684027085445512
  (0, 202)	0.3261751201848034
  (0, 442)	0.33771518675652046
  (0, 1335)	0.30227627373148824
  (0, 717)	0.1931240433675333


In [20]:
sentences = spam_dataset['lemmatized_msg'].apply(lambda x: x.split(" "))

In [21]:
sentences.shape

(5573,)

In [22]:
w2v = Word2Vec(sentences)

In [23]:
w2v.wv.__dir__()

['vector_size',
 'index_to_key',
 'next_index',
 'key_to_index',
 'vectors',
 'norms',
 'expandos',
 'mapfile_path',
 'vectors_lockf',
 '__module__',
 '__init__',
 '__str__',
 '_load_specials',
 '_upconvert_old_vocab',
 'allocate_vecattrs',
 'set_vecattr',
 'get_vecattr',
 'resize_vectors',
 '__len__',
 '__getitem__',
 'get_index',
 'get_vector',
 'word_vec',
 'get_mean_vector',
 'add_vector',
 'add_vectors',
 '__setitem__',
 'has_index_for',
 '__contains__',
 'most_similar_to_given',
 'closer_than',
 'words_closer_than',
 'rank',
 'vectors_norm',
 'get_normed_vectors',
 'fill_norms',
 'index2entity',
 'index2word',
 'vocab',
 'sort_by_descending_frequency',
 'save',
 'most_similar',
 'similar_by_word',
 'similar_by_key',
 'similar_by_vector',
 'wmdistance',
 'most_similar_cosmul',
 'rank_by_centrality',
 'doesnt_match',
 'cosine_similarities',
 'distances',
 'distance',
 'similarity',
 'n_similarity',
 '_log_evaluate_word_analogies',
 'evaluate_word_analogies',
 'log_accuracy',
 'log_

In [24]:
w2v.wv.word_vec("sale") == w2v.wv.get_vector("sale")

/tmp/ipykernel_18/1571672212.py:1: DeprecationWarning: Call to deprecated `word_vec` (Use get_vector instead).
  w2v.wv.word_vec("sale") == w2v.wv.get_vector("sale")


array([ True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True,  True,  True,  True,  True,  True,  True,  True,  True,
        True])

In [25]:
w2v.wv.get_vector("sale")

array([-0.04057318,  0.09556414,  0.01153715, -0.01416206,  0.05590406,
       -0.1583303 ,  0.05003724,  0.19673765, -0.07413613, -0.02959237,
       -0.05102929, -0.07824882, -0.03142149,  0.02749333, -0.01736448,
       -0.08373532,  0.00642818, -0.11106173, -0.01963324, -0.18912569,
        0.06259684,  0.01540956,  0.05431125, -0.03956249, -0.0164717 ,
        0.01617389, -0.07266036, -0.09408571, -0.08329018, -0.01057289,
        0.085933  ,  0.07273242,  0.03458877, -0.03505729, -0.05965972,
        0.06087065, -0.04872144, -0.08397283, -0.05455034, -0.18362263,
        0.05620351, -0.08429234, -0.01748542, -0.00977164,  0.07420126,
       -0.00851146, -0.09051185, -0.00677447,  0.08487814,  0.05031484,
        0.05418573, -0.12624477, -0.03672968, -0.00220218, -0.08428143,
        0.05367894,  0.07151145,  0.02128879, -0.13496415,  0.03626819,
        0.05459635,  0.0151512 , -0.02616498, -0.03642149, -0.11648822,
        0.05152294,  0.06280319,  0.05288385, -0.12608883,  0.10

In [26]:
vocab = list(w2v.wv.key_to_index.keys())

In [27]:
def avg_word_vector_sentence(sentence):
    avg_vector = np.zeros((100,))
    word_count = 0
    for word in sentence:
        if word in vocab:
            word_vec = w2v.wv.get_vector(word)
            avg_vector +=word_vec
            word_count+=1
    if word_count!=0:
        avg_vector/=word_count
    else:
        return np.zeros((100,))
    
    return avg_vector

def avg_w2vec(sentences):
    transformed=[]
    for sentence in tqdm(sentences):
        count=0
        vector=np.zeros(100)
        for word in sentence.split():
            if word in vocab:
                vector+=w2v.wv.get_vector(word)
                count+=1
        if count!=0:
            vector/=count
            transformed.append(vector)
        else:
            transformed.append([])
    return np.array(transformed)

In [28]:
spam_dataset['lemmatized_msg'].shape

(5573,)

In [29]:
#sentence vector is avg word vector of the words of a sentence
w2v_X = np.array([avg_word_vector_sentence(sentence) for sentence in sentences])
# spam_dataset['sentence_vector'] = avg_w2vec(spam_dataset['lemmatized_msg'])


In [30]:
x_train, x_test, y_train, y_test = train_test_split(w2v_X, Y, test_size=0.2, random_state=42)
classifier = RandomForestClassifier()

classifier.fit(x_train, y_train)
classifier.score(x_test, y_test)

0.9668161434977578